In [1]:
import pandas as pd
import altair as alt

df = pd.read_csv('videogamesales.csv')
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)]

yearly_sales = (
    df.groupby('Year')['Global_Sales']
    .sum()
    .reset_index()
    .rename(columns={'Global_Sales': 'Total_Global_Sales'})
)

brush = alt.selection_interval(encodings=['x'])

line = (
    alt.Chart(yearly_sales)
    .mark_line(point=True, color='#4C78A8', strokeWidth=2.5)
    .encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='Total Global Sales (millions)', scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter(brush)
    .properties(width=700, height=300, title='Global Video Game Sales Over the Years')
)

overview = (
    alt.Chart(yearly_sales)
    .mark_area(color='#4C78A8', opacity=0.3)
    .encode(
        x=alt.X('Year:O', title='', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='', axis=alt.Axis(labels=False))
    )
    .add_params(brush)
    .properties(width=700, height=80, title='Drag to select a year range')
)

alt.vconcat(line, overview).configure_view(strokeWidth=0)

ModuleNotFoundError: No module named 'altair'

In [ ]:
import plotly.express as px

genre_region = (
    df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
    .sum()
    .reset_index()
    .sort_values('NA_Sales', ascending=True)
)

genre_long = genre_region.melt(
    id_vars='Genre',
    value_vars=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales'],
    var_name='Region',
    value_name='Sales'
)

genre_long['Region'] = genre_long['Region'].map({
    'NA_Sales': 'North America',
    'EU_Sales': 'Europe',
    'JP_Sales': 'Japan',
    'Other_Sales': 'Other'
})

fig = px.bar(
    genre_long,
    x='Sales',
    y='Genre',
    color='Region',
    orientation='h',
    color_discrete_map={
        'North America': '#4C78A8',
        'Europe': '#F58518',
        'Japan': '#E45756',
        'Other': '#72B7B2'
    },
    title='Video Game Sales by Genre and Region',
    labels={'Sales': 'Total Sales (millions)', 'Genre': 'Genre'},
    barmode='stack'
)

fig.update_layout(
    legend_title_text='Region',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=13),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.update_xaxes(showgrid=True, gridcolor='#eeeeee')
fig.show()